<a href="https://colab.research.google.com/github/jabeent/BAMBOO/blob/main/Copy_of_GEE_NDVI_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Supplemental Exercise: Calculating NDVI using the Google Earth Engine Python API ##
**Author**: Andrew Larkin <br>
**Version Date**: May 7, 2025 <br>
**Organization**: Created for the [Columbia SHARP Google Earth Engine Bootcamp](https://www.publichealth.columbia.edu/academics/non-degree-special-programs/professional-non-degree-programs/skills-health-research-professionals-sharp-training/trainings/google-earth-engine)

Welcome to our supplemental exercise 'Calculating NDVI using the Google Earth Engine (GEE) Python API'. This notebook will guide you through the GEE bootcamp **Lab 2 Parts 4-7** using the [Python API](https://developers.google.com/earth-engine/tutorials/community/intro-to-python-api) rather than the [JavaScript code editor](https://developers.google.com/earth-engine/guides/playground).

**Goals**

By the end of this notebook you will be able to perform the following activities using the Python API:

1.   Understand important differences between Python and JavaScript syntax
2.   Use your credentials to authorize GEE calls from a Python environment
3.   Load datasets from the data catalog and your assets folder into your GEE workspace
4.   Create variables and write functions using Python syntax
5.   Use the map/redue framework
6.   Export analyses to your cloud drive folder

**Requirements**

To complete this notebook you will need to have

1. A Google Earth Engine [Cloud Project](https://developers.google.com/earth-engine/guides/access).
2. A [Google Drive folder](https://workspace.google.com/products/drive/) with at least 10mb of free space

# Strengths and Limitations of Python and JavaScript #

The GEE officially supports two programming languages: Python and JavaScript (JS). JS is the most popular choice among the GEE community for several reasons. First, the [GEE code editor](https://developers.google.com/earth-engine/guides/playground), only available in JS, is a web-based API interface that provides real time interactivity with the GEE. Features such as mapping datasets, printing out debug statements, and searching through GEE documentation are neatly integrated into the code editor for easy access.

While JS and the code editor is a great option for real time interactive spatial analytics, the [Python GEE library](https://developers.google.com/earth-engine/tutorials/community/intro-to-python-api) is a great option for those who are interested in adding other analysis packages into their methods, such as [NumPy](https://numpy.org/) for big data analytics or [PyTorch](https://www.techtarget.com/searchenterpriseai/definition/PyTorch) for deep learning. Python is also the preferred programming language for performing large [asynchronous batch tasks](https://developers.google.com/earth-engine/guides/processing_environments) that may require hours or even days to complete.

<div>
<img src="https://drive.google.com/uc?id=1PhuqtU6TzlO_gi1uEMLB1cBKFOJg7fX0" width="600")/>
</div>

# Setup Python Workspace #

To get started using GEE in Python, you will need to:

1.   Install and import the [GEE library](https://developers.google.com/earth-engine/guides/python_install), called 'ee'. If you are using Google colab then the 'ee' library is already installed.
2.   Authenticate your GEE credentials. Google will use a pop-up window, asking you to sign in to your Google account and set permissions. If you want to export results to your Google drive then you will need to check the permission box that allows Python to 'See, edit, create, and delete all of your Google Drive files'.
3. Initialize your workspace using your assigned project name. Your project name is displayed in the top right corner of the GEE code editor.

<div>
<img src="https://drive.google.com/uc?id=1UjKIiZ6wX-Gd0HPda2AGdHyWObTsWzVt" width="600")/>
</div>



In [ ]:
import ee
import time
ee.Authenticate()
projectName = 'ee-larkinan' # change to your project, where you are storing the NYC datasets
ee.Initialize(project=projectName)

Next, load your custom datasets into the Python workspace. To load the dataset you will need the dataset filepath. You can find the filepath in the GEE code editor by navigating to the dataset in the asset tab and left clicking on it.

<div>
<img src="https://drive.google.com/uc?id=1WpZlvs3gNb5JiBz5_w6hS5h_kEfMNvMi" width="600")/>
</div>

In [ ]:
NYC = ee.FeatureCollection('projects/' + projectName + '/assets/NYC')
NYC_CT = ee.FeatureCollection('projects/' + projectName + '/assets/NYC_CT')
print("number of census tracts in NYC: %i" %(NYC_CT.size().getInfo()))

number of census tracts in NYC: 2165


# Part 4: Get Image Collections for L5, L7, L8, and L9



*   Functions in Python use spacing rather than brackets to denote when the function starts and ends (see example below)
*   commands that are writtten on multiple lines require a \ at the end of each line, except for the final line
*   Unlike JavaScript, when creating a variable in Python it does not need to be preceeded with the keyword 'var'. Semicolons are also not necessary at the end of a command
*   Unlike JavaScript, Python does not automatically convert integrers to strings. For example, in the filterDate function below, the variable 'year' has to be explicltly converted to strings using str(year)
<br>
<br>

<div>
<img src="https://drive.google.com/uc?id=1Nz4QA-_VwzHy_okRObjpE4w6pwb65Anp" width="600")/>
</div>
<br>

In [ ]:
# Get ImageCollections L8, L7, and L5, filtered by date, area of interest, and masking out clouds and shadows
def collectionStack(year, startDay, endDay, aoi):

    # get Landsat 9 image collection for area of interest and a specific time period
    lc9Col = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')\
    .filterDate(str(year) + '-' + str(startDay), str(year) + '-' + str(endDay))\
    .map(maskCloudsAndShadows)

    # get Landsat 8 image collection for area of interest and a specific time period
    lc8Col = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')\
    .filterDate(str(year) + '-' + str(startDay), str(year) + '-' + str(endDay))\
    .map(maskCloudsAndShadows)

    # get Landsat 7 image collection for area of interest and a specific time period
    le7Col  = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2')\
    .filterDate(str(year) + '-' + str(startDay), str(year) + '-' + str(endDay))\
    .map(maskCloudsAndShadows)

    # get Landsat 5 image collection for area of interest and a specific time period
    lt5Col  = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')\
    .filterDate(str(year) + '-' + str(startDay), str(year) + '-' + str(endDay))\
    .map(maskCloudsAndShadows)

    # harmonize Landsat 5, 7 and 8 collection
    lc9Col_harmonized = lc9Col.select(['SR_B4','SR_B5'],['SR_B3', 'SR_B4']) #select B4 and B5 and rename them as B3 and B4 for harmonization purpose
    lc8Col_harmonized = lc8Col.select(['SR_B4','SR_B5'],['SR_B3', 'SR_B4']) #select B4 and B5 and rename them as B3 and B4 for harmonization purpose
    lt7Col_harmonized = le7Col.map(harmonization2OLI) # hormonize landsat 7 to oli
    lt5Col_harmonized = lt5Col.map(harmonization2OLI) # hormonize landsat 5 to oli

    #combine all available landsat images
    combinedCollection = lt5Col_harmonized.merge(lt7Col_harmonized)\
    .merge(lc8Col_harmonized)\
    .merge(lc8Col_harmonized)

    return combinedCollection # return the combined image collection

In [ ]:
# Function to harmonize TM and ETM+ to OLI
# Reference: Roy et al (2016) Characterization of Landsat-7 to Landsat-8 reflective wavelength and normalized difference vegetation index continuity.Romote Sensing of Environment https://doi.org/10.1016/j.rse.2015.12.024)
# More reference: Landsat ETM+ to OLI Harmonization tutorial (https://developers.google.com/earth-engine/tutorials/community/landsat-etm-to-oli-harmonization)
def harmonization2OLI(image):
    #define coefficients; refer to Roy et al. (2016) for the transformation function
    slopes = ee.Image.constant([0.9825, 1.0073])
    itcps = ee.Image.constant([-0.0022, -0.0021])

    # apply harmonization transformation, and add the time information as a image property
    img = image.select(['SR_B3','SR_B4'])\
    .multiply(slopes)\
    .add(itcps)\
    .set('system:time_start', image.get('system:time_start'))

    return img

**Note**: 'and' is a special reserved keyword in Python. To use the GEE and() relational operator in Python, [use 'And()' with an upercase A.](https://developers.google.com/earth-engine/guides/image_relational#colab-python)

In [ ]:
# Mask function for Landsat8 SR (available from https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C01_T1_SR)
# Step 4.2 Create a function to mask pixels with shadows and/or clouds
def maskCloudsAndShadows(image):
    cloudBit = (1 << 3)
    cloudShadowBit = (1 << 4)
    snowBit = (1 << 5)
    waterBit = (1 << 7)

    # select the Landsat 8 QA band for image analysis
    qa = image.select('QA_PIXEL')

    # which pixels in the qa band do not have a red flag for shadows or clouds?
    mask = qa.bitwiseAnd(cloudBit).eq(0)\
    .And(qa.bitwiseAnd(cloudShadowBit).eq(0))\
    .And(qa.bitwiseAnd(waterBit).eq(0))\
    .And(qa.bitwiseAnd(snowBit).eq(0))

    # using the mask, screen the landsat 8 image to only select pixels free from shadows or clouds
    maskedImage = image.updateMask(mask)

    #return the image with shadows and cloud pixels removed
    return(maskedImage)

# Part 5: Get Annual NDVI Composite #

In [ ]:
# function to get annual NDVI composites for the study area
def annualComposite(startYear, endYear, startDay, endDay, aoi):

    imgs = []

    #create a for loop to get the yearly ndvi composite
    for year in range(startYear,endYear+1):
        collection = collectionStack(year, startDay, endDay, aoi) # get all available landsat images
        ndvi_imgs = collection.map(ndvicalc) #calcuate ndvi and return ndvi for each image
        #get the mean ndvi composite for a specific year and concatenate multiple year ndvi to an image collection
        newDate = ee.Date.fromYMD(year,8,1).millis()
        ndvi_mean = ndvi_imgs.mean()\
        .set('system:time_start', newDate)\
        .set('system:index', str(year))
        imgs.append(ndvi_mean) #concatenate multiple year ndvi to a image collection
    return ee.ImageCollection(imgs) # return multiple-year ndvi images

In [ ]:
# function to calculate ndvi
def ndvicalc(image):
    ndvi = image.normalizedDifference(['SR_B4', 'SR_B3'])\
    .select([0], ['NDVI'])\
    .set('system:time_start', image.get('system:time_start'))
    return ndvi

# Part 6: NDVI Time Series Analysis

In [ ]:
# Step 1: Define the start and end year (inclusive) and season for NDVI time series analysis
startYear = 2020
endYear = 2024
startDay = '06-15'
endDay =   '09-15'
aoi = NYC #Define your study area

In [ ]:
# Step 2: Apply the NDVI time series function to get the annual composite
NDVIresult = annualComposite(startYear, endYear, startDay, endDay, aoi) #Call the function
print(NDVIresult.getInfo()) #Print the returned image collection to check its features

{'type': 'ImageCollection', 'bands': [], 'features': [{'type': 'Image', 'bands': [{'id': 'NDVI', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}], 'properties': {'system:time_start': 1596240000000, 'system:index': '2020'}}, {'type': 'Image', 'bands': [{'id': 'NDVI', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}], 'properties': {'system:time_start': 1627776000000, 'system:index': '2021'}}, {'type': 'Image', 'bands': [{'id': 'NDVI', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}], 'properties': {'system:time_start': 1659312000000, 'system:index': '2022'}}, {'type': 'Image', 'bands': [{'id': 'NDVI', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 

# Part 7: Assign Annual NDVI to Census Tracts in NYC #

In [ ]:
# Step 1: Convert the image collection to a multi-band image, and each band is the NDVI composite for a year

# mask the areas outside of NYC to save computing running time (without the masking function, running time will be over an hour)
def clipNYC(img):
    mask = ee.Image.constant(1).clip(NYC).eq(1) # create a mask
    img_masked = img.updateMask(mask)  #apply the mask to the image
    return img_masked #return the masked image

In [ ]:
NDVIresult_masked = NDVIresult.map(clipNYC)
yearlyNDVI = NDVIresult_masked.toBands() # convert the image collection to a multi-band image
print(yearlyNDVI.getInfo())

{'type': 'Image', 'bands': [{'id': '2020_NDVI', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': '2021_NDVI', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': '2022_NDVI', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': '2023_NDVI', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': '2024_NDVI', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}]}


In [ ]:
# Step 2: Develop a function to get the mean NDVI for a region
# Check reduceRegion function: https://developers.google.com/earth-engine/guides/reducers_reduce_region
def AddNDVI(feature):
    NDVI_mean = yearlyNDVI.reduceRegion(
        reducer = ee.Reducer.mean(),
        geometry = feature.geometry(),
        scale = 30,
        maxPixels = 1e9
    )
    return feature.set(NDVI_mean) #append the multiple-year NDVI values extracted from the image to the attribute table as new properties

In [ ]:
# Step 3: Load the NYC census tract and test the function on one census tract
print(NYC_CT.first().getInfo())
NDVI_oneCT = AddNDVI(NYC_CT.first()) # extract the first NYC census tract as an example to test the function
print(NDVI_oneCT.getInfo()) # test the function on one Census Tract feature

{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-73.92727140480648, 40.86554305302726], [-73.92662036085326, 40.864530881146955], [-73.92601389592888, 40.86358108853141], [-73.9253763028967, 40.86259118592214], [-73.9247653490277, 40.86164139849826], [-73.9247431031304, 40.86160121575628], [-73.92462263437619, 40.86164579886904], [-73.9244354277573, 40.86171268615098], [-73.92441309212086, 40.86172161264553], [-73.92387802279356, 40.861917857682116], [-73.92361941289887, 40.86201149658523], [-73.9232180869766, 40.86215865392542], [-73.92307088608969, 40.862212135283556], [-73.92294602089892, 40.86225669033498], [-73.92277217827987, 40.862319160029266], [-73.92249571616803, 40.862421683016954], [-73.9220720520193, 40.86257775918589], [-73.92061393343414, 40.86312181199117], [-73.92125163179766, 40.86412061243056], [-73.92188475324032, 40.865110543952], [-73.92250011157782, 40.8660737049149], [-73.9231110725708, 40.86702351098259], [-73.9250061965833, 40.86634576300

In [ ]:
# Step 4: Load the NYC census tract and use a map function to append mean NDVI to the attribute table
NDVI_CT = NYC_CT.map(AddNDVI) # apply the function to all census tracts

# select which variables to export
selectors = ['OBJECTID']
for year in range(startYear,endYear+1):
    selectors.append(str(year) + "_NDVI")

# Step 5: Run a task to export the table (running time should be less than 5mins; it was ~40s on our end)
task = ee.batch.Export.table.toDrive(
    collection = NDVI_CT, # export the feature collection 'NDVI_CT'
    description = 'Lab2_NDVI', # give the task a description
    fileFormat = 'CSV', # set the table format as csv
    selectors = selectors
)

task.start()
index = 0
while(task.status()['state']!='COMPLETED'):
    print('waiting for %i minutes' %(index))
    time.sleep(60)
    index+=1
print("task completed")

waiting for 0 minutes


KeyboardInterrupt: 